In [4]:
import numpy as np
from qiskit.circuit.library import CXGate, RXXGate, SwapGate, iSwapGate
from qiskit.quantum_info import Operator, average_gate_fidelity
from qutip import Qobj
from weylchamber import c1c2c3

from gulps.utils.invariants import GateInvariants
from gulps.local_numerics import SegmentNumericSynthesizer

In [5]:
# Example usage
# Define a gate list and invariant list for testing
gate_list = [
    GateInvariants.from_unitary(gate) for gate in [CXGate(), CXGate(), CXGate()]
]
invariant_list = [
    GateInvariants.from_weyl(c)
    for c in [(0.5, 0.0, 0.0), (0.5, 0.5, 0), (0.5, 0.5, 0.5)]
]

In [6]:
# Synthesize segments local rotations
segment_solutions = SegmentNumericSynthesizer._synthesize_segments(
    gate_list, invariant_list
)
print("Segment solutions:", segment_solutions)

Segment solutions: [array([ 0.70127154,  4.58352926, -0.70072199,  3.05288158,  2.49471306,
        2.04311779]), array([ 9.91573205,  9.07515997, -9.91573276,  0.03703351,  3.97905326,
       -4.86252828])]


In [7]:
# Recover unitary equivalence by promoting local equivalence
ret = SegmentNumericSynthesizer._stitch_segments(
    gate_list, invariant_list, segment_solutions
)


print(c1c2c3(Operator(ret).data))
ret.draw()

(0.5, 0.5, 0.49999998)


global phase: 5π/4
     ┌─────────┐┌─────────┐       ┌──────────────────────────┐      ┌─────────┐»
q_0: ┤ Unitary ├┤ Unitary ├──■────┤ Rv(3.0529,2.4947,2.0431) ├───■──┤ Unitary ├»
     ├─────────┤├─────────┤┌─┴─┐┌─┴──────────────────────────┴┐┌─┴─┐├─────────┤»
q_1: ┤ Unitary ├┤ Unitary ├┤ X ├┤ Rv(0.70127,4.5835,-0.70072) ├┤ X ├┤ Unitary ├»
     └─────────┘└─────────┘└───┘└─────────────────────────────┘└───┘└─────────┘»
«     ┌─────────────────────────────┐     ┌─────────┐
«q_0: ┤ Rv(0.037034,3.9791,-4.8625) ├──■──┤ Unitary ├
«     └┬───────────────────────────┬┘┌─┴─┐├─────────┤
«q_1: ─┤ Rv(9.9157,9.0752,-9.9157) ├─┤ X ├┤ Unitary ├
«      └───────────────────────────┘ └───┘└─────────┘

In [11]:
# Do both steps in one call
target = GateInvariants.from_unitary(SwapGate())
decomp = SegmentNumericSynthesizer.run(gate_list, invariant_list, target=target)
decomp_u = Operator(decomp)
decomp.draw()
print(average_gate_fidelity(SwapGate(), decomp_u))
Qobj(decomp_u.data)

1.0


Quantum object: dims=[[4], [4]], shape=(4, 4), type='oper', dtype=Dense, isherm=False
Qobj data =
[[ 1.00000000e+00-2.67106747e-08j -2.69634991e-16+5.55111512e-17j
  -1.66559369e-17+5.39012231e-16j -6.61558164e-10-4.69842523e-16j]
 [ 4.49133225e-17-1.22081687e-16j -6.61558092e-10+5.51926664e-16j
   1.00000000e+00+2.67106733e-08j -6.24655903e-16-2.22044605e-16j]
 [-2.06281307e-16+1.11022302e-16j  1.00000000e+00+2.67106727e-08j
   6.61557989e-10+6.16055101e-16j -6.78918043e-17+5.38880240e-16j]
 [ 6.61558093e-10-6.66606698e-16j  1.02152385e-16-2.66825145e-17j
  -1.07083968e-16+1.66533454e-16j  1.00000000e+00-2.67106746e-08j]]